# Variance and Covariance — Interactive Notebook

**Concept 26-variance-covariance** of the math-foundations map.

We explore second-order moments: variance (spread of a single RV) and covariance (linear association between two RVs). We verify the identity
$$\mathrm{Var}(X+Y) = \mathrm{Var}(X) + \mathrm{Var}(Y) + 2\,\mathrm{Cov}(X, Y)$$
empirically, and demonstrate the classic pitfall: uncorrelated $\ne$ independent.

## 1. Helpers (stdlib only)

Empirical mean / variance / covariance from scratch — no NumPy, no SciPy.

In [ ]:
import math, random
random.seed(0)

def mean(xs):
    return sum(xs) / len(xs)

def variance(xs):
    mu = mean(xs)
    return sum((x - mu) ** 2 for x in xs) / len(xs)

def covariance(xs, ys):
    mx, my = mean(xs), mean(ys)
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / len(xs)

## 2. Bernoulli($p$): $\mathrm{Var}(X) = p(1-p)$

We sample $N = 100{,}000$ Bernoulli$(0.3)$ trials and compare empirical variance against theory.

In [ ]:
N = 100_000
p = 0.3
xs = [1 if random.random() < p else 0 for _ in range(N)]
print(f'mean  = {mean(xs):.5f}   theory = {p}')
print(f'var   = {variance(xs):.5f}   theory = {p*(1-p)}')

## 3. Uniform$[0, 1]$: $\mathrm{Var}(X) = 1/12$

Quick check of a continuous distribution.

In [ ]:
us = [random.random() for _ in range(N)]
print(f'mean  = {mean(us):.5f}   theory = {0.5}')
print(f'var   = {variance(us):.5f}   theory = {1/12:.5f}')

## 4. Correlated Gaussians via Box–Muller

Generate $Z_1, Z_2 \sim \mathcal{N}(0,1)$ independent. Then
$$X = Z_1, \qquad Y = \rho Z_1 + \sqrt{1 - \rho^2}\, Z_2$$
gives $\mathrm{Var}(X) = \mathrm{Var}(Y) = 1$ and $\mathrm{Cov}(X, Y) = \rho$.

In [ ]:
def box_muller(n):
    out = []
    while len(out) < n:
        u1 = random.random() or 1e-12
        u2 = random.random()
        r = math.sqrt(-2.0 * math.log(u1))
        out.append(r * math.cos(2*math.pi*u2))
        if len(out) < n:
            out.append(r * math.sin(2*math.pi*u2))
    return out

rho = 0.6
z1 = box_muller(N)
z2 = box_muller(N)
X = z1
Y = [rho*a + math.sqrt(1 - rho*rho)*b for a, b in zip(z1, z2)]

vX, vY = variance(X), variance(Y)
cXY = covariance(X, Y)
corr = cXY / math.sqrt(vX*vY)
print(f'Var(X)   = {vX:.4f}     (theory 1.0)')
print(f'Var(Y)   = {vY:.4f}     (theory 1.0)')
print(f'Cov(X,Y) = {cXY:.4f}    (theory {rho})')
print(f'corr_emp = {corr:.4f}    (theory {rho})')

## 5. Verify the variance-of-sum identity

$$\mathrm{Var}(X + Y) \;\stackrel{?}{=}\; \mathrm{Var}(X) + \mathrm{Var}(Y) + 2\,\mathrm{Cov}(X, Y)$$

In [ ]:
S = [x + y for x, y in zip(X, Y)]
vS = variance(S)
rhs = vX + vY + 2*cXY
print(f'Var(X+Y) direct  = {vS:.6f}')
print(f'Var(X+Y) formula = {rhs:.6f}')
print(f'abs diff         = {abs(vS - rhs):.2e}')

## 6. Uncorrelated does NOT imply independent

Take $X \sim \mathrm{Uniform}(\{-1, 0, 1\})$ and $Y = X^2$. Then $\mathrm{Cov}(X, Y) = E[X^3] - E[X]E[X^2] = 0$, yet $Y$ is a deterministic function of $X$.

In [ ]:
X3 = [random.choice([-1, 0, 1]) for _ in range(N)]
Y3 = [x*x for x in X3]
print(f'Cov(X, X^2) = {covariance(X3, Y3):.5f}   (theory 0)')
print(f'But Y is a function of X: knowing X determines Y exactly.')

## 7. GRPO-style advantage normalization

Group Relative Policy Optimization normalizes rewards across a rollout group:
$$\hat A_i = \frac{r_i - \mu_r}{\sigma_r}.$$
The resulting advantages have sample mean $0$ and sample variance $1$ — variance is what makes this scale-invariant.

In [ ]:
rewards = [0.1, 0.4, 0.5, 0.9, 1.0]
mu = mean(rewards)
sigma = math.sqrt(variance(rewards))
A = [(r - mu) / sigma for r in rewards]
print(f'mu       = {mu:.4f}')
print(f'sigma    = {sigma:.4f}')
print(f'A_hat    = {[round(a, 4) for a in A]}')
print(f'mean(A)  = {mean(A):.2e}    (≈ 0)')
print(f'var(A)   = {variance(A):.4f}   (= 1)')